# TASK3 Step 2 — full E1 SFT and validation
Run only after the User signs GO for P4. This notebook stops after validation; it never builds a submission ZIP or promotes the adapter.

In [ ]:
!pip install -q -e '/kaggle/input/legalqa-code-v3[training,eval]'

In [ ]:
import json
import os
from pathlib import Path

import torch

from legalqa.core.config import load_config
from legalqa.core.manifest import code_hash, config_hash, sha256_file

CODE = Path('/kaggle/input/legalqa-code-v3')
DATA = Path('/kaggle/input/task2-data-v3')
INDEX = Path('/kaggle/input/task2-index-v3')
SFT = Path('/kaggle/input/task2-e1-sft-v1/train_sft.jsonl')
WORK = Path('/kaggle/working/legalqa-runtime')
INDEX_MOUNT = WORK / 'Data/Task2/indexes/task2-data-v3-bm25-candidate/index'
SFT_MOUNT = WORK / 'Data/Task2/runs/e1-full-v1/train_sft.jsonl'
WORK.mkdir(parents=True, exist_ok=True)
INDEX_MOUNT.parent.mkdir(parents=True, exist_ok=True)
SFT_MOUNT.parent.mkdir(parents=True, exist_ok=True)
os.symlink(INDEX, INDEX_MOUNT, target_is_directory=True)
os.symlink(SFT, SFT_MOUNT)
os.chdir(WORK)

expected = json.loads((CODE / 'expected_hashes_step2.json').read_text())
config = load_config(CODE / 'configs/e1_rag_step2.yaml')
assert torch.cuda.is_available(), 'NO_CUDA_DEVICE'
assert code_hash() == expected['code_hash'], 'CODE_HASH_MISMATCH'
assert config_hash(config) == expected['config_hash'], 'CONFIG_HASH_MISMATCH'
assert sha256_file(DATA / 'manifest.json') == expected['release_manifest_sha256'], 'RELEASE_HASH_MISMATCH'
assert sha256_file(SFT) == expected['input_sha256'], 'SFT_HASH_MISMATCH'
assert sha256_file(INDEX / 'index_manifest.json') == expected['index_manifest_sha256'], 'INDEX_MANIFEST_HASH_MISMATCH'
assert sha256_file(INDEX / 'bm25.sqlite3') == expected['index_artifact_sha256'], 'INDEX_ARTIFACT_HASH_MISMATCH'
os.environ['LEGALQA_GIT_COMMIT'] = expected['git_commit']

In [ ]:
from legalqa.training import train_e1

training_manifest = train_e1(config, progress=print)
assert training_manifest['adapter_sha256']
training_manifest

In [ ]:
from legalqa.core.io import write_json
from legalqa.evaluation import evaluate_qa_files
from legalqa.generation.pipeline import LegalQAPipeline

adapter = Path(config.training.output_dir)
predictions = Path('/kaggle/working/step2_validation_predictions.json')
pipeline = LegalQAPipeline(config)
pipeline.predict_file(
    DATA / 'qa/validation.jsonl', predictions,
    trace_path='/kaggle/working/step2_trace.json', progress=print,
    environment='kaggle_gpu', adapter_path=adapter,
)
manifest = json.loads(Path(f'{predictions}.manifest.json').read_text())
assert manifest['adapter_sha256'] == training_manifest['adapter_sha256'], 'TRAIN_INFER_ADAPTER_MISMATCH'
metrics = evaluate_qa_files(predictions, DATA / 'qa/validation.jsonl')
write_json('/kaggle/working/step2_metrics.json', metrics)
metrics